In [1]:
!pip install grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 84.3 MB/s eta 0:00:00:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for grad-cam: filename=grad_cam-1.5.5-py3-none-any.whl size=44284 sha256=7b51fcd73aaaa9c62b8f58c3e764f6dd23aadfb5cc3f28b2cd4c3dbb7336b3da
  Stored in directory: /root/.cache/pip/wheels/fb/3b/09/2afc520f3d69bc26ae6bd87416759c820a3f7d05c1a077bbf6
Successfully built grad-cam


In [2]:
import os
import numpy as np
import timm
import matplotlib.pyplot as plt
import torch.nn.functional as F

import torch
from torch import nn
from torchvision import models
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [3]:
# Check accelerator

device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print(f'Using {device} device')

Using cuda device


In [4]:
# Path diretory of dataset
TRAIN_DIR = '/kaggle/input/brain-tumor-mri-dataset/Training'
TRAIN_DIR

'/kaggle/input/brain-tumor-mri-dataset/Training'

In [5]:
# Path diretory of testing data
TEST_DIR = '/kaggle/input/brain-tumor-mri-dataset/Testing'
TEST_DIR

'/kaggle/input/brain-tumor-mri-dataset/Testing'

In [6]:
# DenseNet121
MODEL1_DIR = '/kaggle/input/densenet-mri-train-test-42'

# Mobilenet
MODEL2_DIR = '/kaggle/input/mobilenet-mri-train-test-42'

In [7]:
# Model name

MODEL1_NAME = 'DenseNet121'
MODEL2_NAME = 'MobileNet'

In [8]:
# Hyperparameters

BATCH_SIZE = 32
EPOCHS = 50
NUM_CLASSES = 4
DROPOUT_RATE = 0.3
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

In [9]:
def get_data_loaders(train_dir: str = TRAIN_DIR, test_dir: str = TEST_DIR, batch_size: int = BATCH_SIZE) -> tuple[DataLoader, DataLoader]:
    """
    Creates PyTorch DataLoaders from train and test directories.

    Args:
        train_dir (str): Path to the training dataset directory.
        test_dir (str): Path to the test/validation dataset directory.
        batch_size (int): Batch size.

    Returns:
        tuple[DataLoader, DataLoader]: (train_loader, val_loader)
    """
    
    # Standard ImageNet normalization statistics (Required for weights)
    norm_mean=[0.485, 0.456, 0.406]
    norm_std=[0.229, 0.224, 0.225]

    # Training Transform Pipeline
    train_transform = v2.Compose([
        # Resize to 256x256 first. This provides a buffer for subsequent 
        # rotation/translation and cropping, preventing black border artifacts.
        v2.Resize(size=256),

        # Apply Data Augmentation
        v2.RandomHorizontalFlip(),
        v2.RandomRotation(degrees=36),
        v2.RandomAffine(degrees=0, scale=(0.9, 1.1)),
        v2.ColorJitter(brightness=0.1, contrast=0.1),

        # Use CenterCrop to focus on the primary subject
        v2.CenterCrop(size=224),

        # Convert PIL/Numpy to Tensor, cast to Float32, and rescale to [0, 1]
        v2.ToImage(),
        v2.ToDtype(dtype=torch.float32, scale=True),

        # Normalize using ImageNet mean and std
        v2.Normalize(mean=norm_mean, std=norm_std),
    ])

    # Validation Transform Pipeline
    val_transform = v2.Compose([
        v2.Resize(size=256),
        v2.CenterCrop(size=224),
        v2.ToImage(),
        v2.ToDtype(dtype=torch.float32, scale=True),
        v2.Normalize(mean=norm_mean, std=norm_std)
    ])

    # Worker Configuration
    # Determine the optimal number of CPU workers to prevent bottlenecks.
    # Capped at 4 to avoid excessive memory overhead.
    num_workers = min(4, os.cpu_count())

    train_dataset = ImageFolder(root=train_dir, transform=train_transform)
    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)

    val_dataset = ImageFolder(root=test_dir, transform=val_transform)
    val_loader = DataLoader(dataset=val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

    return train_loader, val_loader

In [10]:
class DenseNet121(nn.Module):
    """
    DenseNet121-based model for image classification using Transfer Learning.
    
    Architecture:
    - Backbone: DenseNet121 (frozen, ImageNet weights)
    - Head: Global Average Pooling -> Dropout -> Linear (Dense)
    
    Attributes:
        num_classes (int): Number of target classes.
        dropout_rate (float): Dropout probability.
    """
    
    def __init__(self, num_classes: int = NUM_CLASSES, dropout_rate: float = 0.3) -> None:
        super().__init__()

        # Load Pre-trained DenseNet121
        weights = models.DenseNet121_Weights.IMAGENET1K_V1
        backbone = models.densenet121(weights=weights)

        # DenseNet121 .features contains all Conv/Relu/MaxPool layers
        self.features = backbone.features

        # Freezing parameters to prevent updating during training
        for param in self.features.parameters():
            param.requires_grad = False

        # Define Custom Classifier Head.
        self.in_features = 1024 
        self.global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1) # Reduces (Batch, 1024, H, W) -> (Batch, 1024, 1, 1)
        self.flatten = nn.Flatten(start_dim=1) # Flatten (Batch, 1024, 1, 1) -> (Batch, 1024)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(in_features=self.in_features, out_features=num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            x (torch.Tensor): Input image batch (B, C, H, W).
                              Expected standard ImageNet normalization.

        Returns:
            torch.Tensor: Raw logits (B, num_classes). 
                          NOTE: Does NOT apply Softmax.
        """
        
        # Feature extraction (Frozen)
        x = self.features(x)
        # The torchvision.models.densenet121 `.features` block ends with a 
        # BatchNorm layer (norm5), which outputs both negative and positive values.
        # We MUST apply ReLU here to zero out negative values (noise/background).
        x = F.relu(x, inplace=True)
        # Classification Head (Trainable)
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        
        # Output Logits
        logits = self.classifier(x)
        
        return logits

In [11]:
def build_densenet121(num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> DenseNet121:
    """
    Factory function to instantiate the customized DenseNet121 model for Transfer Learning.

    This function initializes a `DenseNet121` which includes:
    1. A frozen DenseNet121 backbone (pre-trained on ImageNet).
    2. A custom trainable classification head (ReLU -> Pooling -> Dropout -> Linear).

    Args:
        num_classes (int, optional): The number of target classes for the classification task. 
            Determines the output size of the final Linear layer. Defaults to 3.
        dropout_rate (float, optional): The probability of an element to be zeroed 
            in the Dropout layer. Used for regularization to prevent overfitting. 
            Defaults to 0.3.

    Returns:
        DenseNet121: An initialized PyTorch model instance (subclass of nn.Module), 
            ready to be moved to a device (CPU/GPU) for training or inference.
    """
    
    model = DenseNet121(num_classes=num_classes, dropout_rate=dropout_rate)
    return model

In [12]:
class MobileNet(nn.Module):
    """
    MobileNetV1-based model for image classification using Transfer Learning.
    
    Architecture:
    - Backbone: MobileNetV1 (frozen, ImageNet weights, via timm)
    - Head: Global Average Pooling -> Dropout -> Linear (Dense)
    
    Attributes:
        num_classes (int): Number of target classes.
        dropout_rate (float): Dropout probability.
    """

    def __init__(self, num_classes: int = NUM_CLASSES, dropout_rate: float = 0.3) -> None:
        super().__init__()
        
        # Load Backbone: MobileNet V1 with ImageNet weights.
        # 'mobilenetv1_100' in timm corresponds to MobileNet(alpha=1.0) in Keras.
        self.backbone = timm.create_model('mobilenetv1_100', pretrained=True, num_classes=0, global_pool='')
        
        # Freeze Backbone
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Define Custom Classifier Head
 
        
        # Components matching your MobileNetV1 structure
        self.global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1) # # Reduces (Batch, 1024, H, W) -> (Batch, 1024, 1, 1)
        self.flatten = nn.Flatten(start_dim=1) # Flatten (Batch, 1024, 1, 1) -> (Batch, 1024)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.classifier = nn.Linear(in_features=1024, out_features=num_classes) # MobileNet features output exactly 1024 channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the model.

        Args:
            x (torch.Tensor): Input image batch (B, C, H, W)

        Returns:
            torch.Tensor: Raw logits (B, num_classes). 
                          NOTE: Does NOT apply Softmax.
        """
        # Feature Extraction (Frozen)
        x = self.backbone(x) 
        
        # Classification Head (Trainable)
        x = self.global_avg_pool(x)
        x = self.flatten(x)
        x = self.dropout(x)
        
        # Output Logits
        logits = self.classifier(x)
        
        return logits

In [13]:
def build_mobilenet(num_classes: int = NUM_CLASSES, dropout_rate: float = DROPOUT_RATE) -> MobileNet:
    """
    Factory function to instantiate the customized MobileNet model for Transfer Learning.

    This function initializes a `MobileNet` which includes:
    1. A frozen mobilenet backbone (pre-trained on ImageNet).
    2. A custom trainable classification head (Pooling -> Dropout -> Linear).

    Args:
        num_classes (int, optional): The number of target classes for the classification task. 
            Determines the output size of the final Linear layer. Defaults to 3.
        dropout_rate (float, optional): The probability of an element to be zeroed 
            in the Dropout layer. Used for regularization to prevent overfitting. 
            Defaults to 0.3.

    Returns:
        MobileNet: An initialized PyTorch model instance (subclass of nn.Module), 
            ready to be moved to a device (CPU/GPU) for training or inference.
    """

    model = MobileNet(num_classes=num_classes, dropout_rate=dropout_rate)
    return model

In [14]:
def visualize_gradcam(densenet: nn.Module, mobilenet: nn.Module, img_tensor: torch.Tensor, img_numpy: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Generates and visualizes Grad-CAM heatmaps for both DenseNet and MobileNet models.

    Args:
        densenet (nn.Module): The trained DenseNet121 model.
        mobilenet (nn.Module): The trained MobileNet model.
        img_tensor (torch.Tensor): Preprocessed image tensor with shape (1, 3, H, W).
        img_numpy (np.ndarray): Original image as a numpy array with shape (H, W, 3), scaled to [0, 1].

    Returns:
        tuple[np.ndarray, np.ndarray]: A tuple containing the Grad-CAM visual results (DenseNet, MobileNet).
    """

    # Set models to evaluation mode to ensure consistent forward pass behavior
    densenet.eval()
    mobilenet.eval()

    # Define target layers for Grad-CAM visualization
    target_layers_densenet = [densenet.features[-1]]
    target_layers_mobilenet = [mobilenet.backbone.blocks[-1][-1].conv_pw]

    # Use None to target the highest-scoring class automatically
    targets = None

    # Generate grayscale CAM mask and overlay heatmap on the original image
    with GradCAM(model=densenet, target_layers=target_layers_densenet) as cam_densenet:
        grayscale_cam_densenet = cam_densenet(input_tensor=img_tensor, targets=targets)[0]
        vis_densenet = show_cam_on_image(img_numpy, grayscale_cam_densenet, use_rgb=True)

    with GradCAM(model=mobilenet, target_layers=target_layers_mobilenet) as cam_mobilenet:
        grayscale_cam_mobilenet = cam_mobilenet(input_tensor=img_tensor, targets=targets)[0]
        vis_mobilenet = show_cam_on_image(img_numpy, grayscale_cam_mobilenet, use_rgb=True)

    return vis_densenet, vis_mobilenet

In [15]:
train_loader, test_loader = get_data_loaders()

In [16]:
model_densenet = build_densenet121()
model_densenet.load_state_dict(torch.load(os.path.join(MODEL1_DIR, f'{MODEL1_NAME}_block_3.pth')))
model_densenet.to(device)

model_mobilenet = build_mobilenet()
model_mobilenet.load_state_dict(torch.load(os.path.join(MODEL2_DIR, f'{MODEL2_NAME}_block_1.pth')))
model_mobilenet.to(device)

Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 245MB/s]


model.safetensors:   0%|          | 0.00/17.0M [00:00<?, ?B/s]

MobileNet(
  (backbone): EfficientNet(
    (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNormAct2d(
      32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU6(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn1): BatchNormAct2d(
            32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): ReLU6(inplace=True)
          )
          (aa): Identity()
          (se): Identity()
          (conv_pw): Conv2d(32, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn2): BatchNormAct2d(
            64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): ReLU6(inplace=

In [17]:
# 1. Define the main output directory
output_dir = 'gradcam_results'
os.makedirs(output_dir, exist_ok=True)

# 2. Get class mapping from the dataset
class_to_idx = test_loader.dataset.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}
target_classes = ['glioma', 'meningioma', 'pituitary'] # We exclude 'notumor'

# Dictionary to keep track of how many images we have saved per class
images_per_class = 5
saved_counts = {cls: 0 for cls in target_classes}

print("Starting to save Grad-CAM results...")

# 3. Iterate through the test loader
for images, labels in test_loader:
    # Check if we have collected enough images for all target classes
    if all(count >= images_per_class for count in saved_counts.values()):
        break
        
    for i in range(len(labels)):
        class_name = idx_to_class[labels[i].item()]
        
        # Process only if it's a target class and we need more samples
        if class_name in target_classes and saved_counts[class_name] < images_per_class:
            
            # Prepare image tensor for Grad-CAM
            img_tensor = images[i].unsqueeze(0).to(device)
            img_tensor.requires_grad = True
            
            # Prepare image numpy for visualization
            img_numpy = images[i].permute(1, 2, 0).cpu().numpy()
            img_numpy = (img_numpy - img_numpy.min()) / (img_numpy.max() - img_numpy.min())
            
            # Generate Grad-CAM visualizations
            cam_dense, cam_mobile = visualize_gradcam(
                densenet=model_densenet,
                mobilenet=model_mobilenet,
                img_tensor=img_tensor,
                img_numpy=img_numpy,
            )
            
            # 4. Create a plot and save it
            plt.figure(figsize=(15, 5))
            
            plt.subplot(1, 3, 1)
            plt.title(f"Original: {class_name}")
            plt.imshow(img_numpy)
            plt.axis("off")
            
            plt.subplot(1, 3, 2)
            plt.title("DenseNet Grad-CAM")
            plt.imshow(cam_dense)
            plt.axis("off")
            
            plt.subplot(1, 3, 3)
            plt.title("MobileNet Grad-CAM")
            plt.imshow(cam_mobile)
            plt.axis("off")
            
            # Save to the class-specific folder
            save_path = os.path.join(output_dir, f"{class_name}_{saved_counts[class_name]}.png")
            plt.savefig(save_path, bbox_inches='tight')
            plt.close() # Close to free up memory
            
            saved_counts[class_name] += 1
            print(f"Saved: {save_path}")

print("\nDone! All images are saved in the 'gradcam_results' folder.")

Starting to save Grad-CAM results...
Saved: gradcam_results/glioma_0.png
Saved: gradcam_results/glioma_1.png
Saved: gradcam_results/glioma_2.png
Saved: gradcam_results/glioma_3.png
Saved: gradcam_results/glioma_4.png
Saved: gradcam_results/meningioma_0.png
Saved: gradcam_results/meningioma_1.png
Saved: gradcam_results/meningioma_2.png
Saved: gradcam_results/meningioma_3.png
Saved: gradcam_results/meningioma_4.png
Saved: gradcam_results/pituitary_0.png
Saved: gradcam_results/pituitary_1.png
Saved: gradcam_results/pituitary_2.png
Saved: gradcam_results/pituitary_3.png
Saved: gradcam_results/pituitary_4.png

Done! All images are saved in the 'gradcam_results' folder.


In [18]:
!zip -r gradcam_results.zip gradcam_results

  adding: gradcam_results/ (stored 0%)
  adding: gradcam_results/pituitary_4.png (deflated 0%)
  adding: gradcam_results/pituitary_1.png (deflated 0%)
  adding: gradcam_results/meningioma_1.png (deflated 0%)
  adding: gradcam_results/meningioma_0.png (deflated 0%)
  adding: gradcam_results/glioma_3.png (deflated 0%)
  adding: gradcam_results/glioma_0.png (deflated 0%)
  adding: gradcam_results/meningioma_2.png (deflated 0%)
  adding: gradcam_results/glioma_2.png (deflated 1%)
  adding: gradcam_results/meningioma_4.png (deflated 0%)
  adding: gradcam_results/glioma_1.png (deflated 0%)
  adding: gradcam_results/pituitary_0.png (deflated 0%)
  adding: gradcam_results/pituitary_2.png (deflated 0%)
  adding: gradcam_results/meningioma_3.png (deflated 0%)
  adding: gradcam_results/glioma_4.png (deflated 0%)
  adding: gradcam_results/pituitary_3.png (deflated 0%)
